# Módulo 3 — t-SNE / UMAP **3D** del Agent OS (Manuelita S.A.)
### Bonus "picante" (Ruta Transversal B) · visualización interactiva con `plotly.graph_objects`

Este notebook proyecta en **3D interactivo** dos espacios reales del agente
`manuelita-bot` (OpenFang Agent OS), extraídos de su base de datos nativa
`openfang.db`:

- **Panel A — Espacio de conocimiento:** corpus OSINT desplegado + 20 hechos
  núcleo etiquetados (verdad-de-terreno). Embeddings locales multilingües.
- **Panel B — Memoria semántica real del OS:** los vectores **768-dim** que
  OpenFang guardó (capa 2 de su modelo de memoria de 6 capas), limpiados y
  etiquetados por verdad-de-terreno.

Reducción dimensional con **t-SNE** y **UMAP**; clustering con **KMeans**;
calidad medida con **silhouette** y **ARI** (Adjusted Rand Index).

> Reutiliza la lógica de `scripts/tsne_sesiones_m3.py` y
> `scripts/tsne_probe_facts.py` (no duplica código). No requiere el daemon
> encendido: lee la copia `reports/modulo3/_work/openfang.db`.

## 0. Setup — imports y rutas

In [1]:
import os, sys
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"   # incrusta plotly.js vía CDN

# Localizar la raíz del repo (robusto al cwd) y cargar la lógica ya probada.
REPO = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(REPO, "scripts", "tsne_probe_facts.py")):
        break
    REPO = os.path.dirname(REPO)
sys.path.insert(0, os.path.join(REPO, "scripts"))

import tsne_sesiones_m3 as T                      # conectar, reducir, métricas...
from tsne_probe_facts import PROBE_FACTS, clean_memory_content, match_theme
from sklearn.cluster import KMeans

SEED = 42
print("REPO:", REPO)
print("DB  :", T.DEFAULT_DB)
print("Hechos etiquetados (verdad-de-terreno):", len(PROBE_FACTS))

REPO: C:\Users\PROYECTOS\Desktop\Claude_Multi_Agents_Projects\proyecto_manuelita
DB  : C:\Users\PROYECTOS\Desktop\Claude_Multi_Agents_Projects\proyecto_manuelita\reports\modulo3\_work\openfang.db
Hechos etiquetados (verdad-de-terreno): 20


## 1. Extracción de datos reales del `openfang.db`

In [2]:
con = T.conectar(T.DEFAULT_DB)
mem_raw, mem_vecs = T.leer_memorias_reales(con)
sesiones = T.leer_sesiones(con)
n_use, tin, tout, cost = T.leer_costos(con)

print(f"Memorias con embedding : {mem_vecs.shape[0]}  (dim={mem_vecs.shape[1] if mem_vecs.shape[0] else 0})")
print(f"Sesiones con mensajes  : {len(sesiones)}")
print(f"Costo acumulado (real) : ${cost:.4f}  en {n_use} llamadas "
      f"(input {tin:.0f} / output {tout:.0f} tokens)")

Memorias con embedding : 21  (dim=768)
Sesiones con mensajes  : 1
Costo acumulado (real) : $0.1855  en 21 llamadas (input 181770 / output 1228 tokens)


## 2. Panel A — Espacio de conocimiento (corpus + 20 hechos)

Embebemos el corpus desplegado + los 20 hechos núcleo con un
sentence-transformer multilingüe (`paraphrase-multilingual-MiniLM-L12-v2`,
384-dim, apropiado para español). Clusterizamos con KMeans y medimos calidad.

In [3]:
from sentence_transformers import SentenceTransformer
modelo = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

textos, temas, fuentes = T.construir_conocimiento()
X_know = np.asarray(modelo.encode(textos, show_progress_bar=False), dtype=np.float32)

k = len(set(temas))
km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_know)
pureza_A, rows_A = T.pureza_clusters(km.labels_, temas, k)
sil_A, ari_A = T.metricas_duras(X_know, km.labels_, temas)

print(f"{len(textos)} ítems · {k} temas")
print(f"Pureza KMeans : {pureza_A:.0%}")
print(f"silhouette    : {sil_A:.3f}")
print(f"ARI           : {ari_A:.3f}")

C:\Users\PROYECTOS\Desktop\Claude_Multi_Agents_Projects\proyecto_manuelita\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8819.85it/s]

104 ítems · 6 temas
Pureza KMeans : 58%
silhouette    : 0.091
ARI           : 0.228


### 2.1 Reducción 3D (t-SNE y UMAP) y figura interactiva

In [4]:
know_tsne3, perp_k = T.tsne_reducir(X_know, n_components=3)
know_umap3, nn_k    = T.umap_reducir(X_know, n_components=3)

def fig3d(coords, temas_pt, textos_pt, size, title):
    """Scatter 3D con plotly.graph_objects: una traza go.Scatter3d por tema."""
    fig = go.Figure()
    for tema in sorted(set(temas_pt)):
        idx = [i for i, t in enumerate(temas_pt) if t == tema]
        if not idx:
            continue
        fig.add_trace(go.Scatter3d(
            x=coords[idx, 0], y=coords[idx, 1], z=coords[idx, 2],
            mode="markers", name=tema,
            marker=dict(size=size, color=T.TEMA_COLOR.get(tema, "#333"),
                        opacity=0.85, line=dict(width=0.5, color="white")),
            text=[textos_pt[i][:110] for i in idx],
            hovertemplate="<b>%{text}</b><br>tema=" + tema + "<extra></extra>",
        ))
    fig.update_layout(
        title=title, height=620,
        scene=dict(xaxis_title="dim 1", yaxis_title="dim 2", zaxis_title="dim 3"),
        legend=dict(title="Tema"), margin=dict(l=0, r=0, t=60, b=0))
    return fig

fig3d(know_tsne3, temas, textos, 4,
      f"Panel A · Conocimiento · <b>t-SNE 3D</b> · perp={perp_k:.0f} · "
      f"pureza {pureza_A:.0%} · silhouette {sil_A:.3f} · ARI {ari_A:.3f}").show()

C:\Users\PROYECTOS\Desktop\Claude_Multi_Agents_Projects\proyecto_manuelita\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### 2.2 Panel A — misma data, proyección **UMAP 3D** (preserva mejor la estructura global)

In [5]:
fig3d(know_umap3, temas, textos, 4,
      f"Panel A · Conocimiento · <b>UMAP 3D</b> · n_neighbors={nn_k} · "
      f"pureza {pureza_A:.0%} · silhouette {sil_A:.3f} · ARI {ari_A:.3f}").show()

## 3. Panel B — Memoria semántica **real** del OS (768-dim)

Tomamos los vectores nativos que OpenFang guardó en la tabla `memories`. Cada
memoria se **limpia** (se quita el envoltorio `User asked: ... I responded: ...`)
y se etiqueta por **verdad-de-terreno** con `match_theme()`. Las memorias no
etiquetables (conversacionales) se excluyen de las métricas.

> Esto responde la pregunta del equipo: **¿la memoria del Agent OS clusteriza por
> tema?** — medido sobre los embeddings nativos del OS, no sobre un proxy.

In [6]:
mem_clean  = [clean_memory_content(c) for c in mem_raw]
temas_mem  = [match_theme(c) or "Conversacional" for c in mem_raw]
idx_lab    = [i for i, t in enumerate(temas_mem) if t != "Conversacional"]
temas_lab  = [temas_mem[i] for i in idx_lab]
textos_lab = [mem_clean[i] for i in idx_lab]

print(f"Memorias totales      : {mem_vecs.shape[0]}")
print(f"Etiquetables (GT)     : {len(idx_lab)}")
print(f"Conversacionales      : {mem_vecs.shape[0] - len(idx_lab)}")
print("Distribución por tema :")
for t in sorted(set(temas_lab)):
    print(f"   {t:<24} {temas_lab.count(t)}")

Memorias totales      : 21
Etiquetables (GT)     : 20
Conversacionales      : 1
Distribución por tema :
   Financiero               4
   Geografía/Operación      4
   Identidad/Corporativo    4
   Productos/Producción     4
   Sostenibilidad           4


In [7]:
sil_B = ari_B = pureza_B = None
if len(idx_lab) >= 3:
    X_mem = mem_vecs[idx_lab]
    k_b = max(2, min(len(set(temas_lab)), len(idx_lab) - 1))
    km_b = KMeans(n_clusters=k_b, n_init=10, random_state=SEED).fit(X_mem)
    pureza_B, rows_B = T.pureza_clusters(km_b.labels_, temas_lab, k_b)
    sil_B, ari_B = T.metricas_duras(X_mem, km_b.labels_, temas_lab)
    memB_tsne3, perp_m = T.tsne_reducir(X_mem, n_components=3)
    try:
        memB_umap3, nn_m = T.umap_reducir(X_mem, n_components=3)
    except Exception as e:
        memB_umap3, nn_m = memB_tsne3, 0
        print("UMAP Panel B no disponible:", e)
    print(f"Panel B (memoria real): pureza {pureza_B:.0%} · silhouette {sil_B:.3f} · ARI {ari_B:.3f}")
    print("NOTA: con pocos puntos estas métricas son indicativas, no concluyentes.")
else:
    print(f"Solo {len(idx_lab)} memorias etiquetables (<3): Panel B sin métricas/3D robustos.")

Panel B (memoria real): pureza 65% · silhouette 0.090 · ARI 0.259
NOTA: con pocos puntos estas métricas son indicativas, no concluyentes.


C:\Users\PROYECTOS\Desktop\Claude_Multi_Agents_Projects\proyecto_manuelita\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


### 3.1 Figura 3D de la memoria real — t-SNE

In [8]:
if len(idx_lab) >= 3:
    fig3d(memB_tsne3, temas_lab, textos_lab, 9,
          f"Panel B · Memoria real del OS ({len(idx_lab)} etiquetadas, "
          f"{mem_vecs.shape[1]}-dim) · <b>t-SNE 3D</b> · "
          f"silhouette {sil_B:.3f} · ARI {ari_B:.3f}").show()
else:
    print("Memoria real insuficiente para la figura 3D del Panel B.")

### 3.2 Figura 3D de la memoria real — UMAP

In [9]:
if len(idx_lab) >= 3:
    fig3d(memB_umap3, temas_lab, textos_lab, 9,
          f"Panel B · Memoria real del OS · <b>UMAP 3D</b> · "
          f"silhouette {sil_B:.3f} · ARI {ari_B:.3f}").show()
else:
    print("Memoria real insuficiente para la figura 3D del Panel B.")

## 4. Resumen de métricas

In [10]:
def _f(v): return f"{v:.3f}" if isinstance(v, float) else "n/d"
print("="*60)
print(f"{'Espacio':<26}{'pureza':>8}{'silhouette':>12}{'ARI':>8}")
print("-"*60)
print(f"{'A) Conocimiento':<26}{pureza_A:>7.0%}{_f(sil_A):>12}{_f(ari_A):>8}")
pB = f"{pureza_B:.0%}" if pureza_B is not None else "n/d"
print(f"{'B) Memoria real (OS)':<26}{pB:>8}{_f(sil_B):>12}{_f(ari_B):>8}")
print("="*60)
print(f"Costo real acumulado: ${cost:.4f} ({n_use} llamadas) · Sesiones: {len(sesiones)}")
con.close()

Espacio                     pureza  silhouette     ARI
------------------------------------------------------------
A) Conocimiento               58%       0.091   0.228
B) Memoria real (OS)           65%       0.090   0.259
Costo real acumulado: $0.1855 (21 llamadas) · Sesiones: 1


## 5. Conclusiones (lectura honesta)

1. **El conocimiento corporativo NO está perfectamente siloado por tema.** La
   pureza/silhouette del Panel A reflejan que un corpus **mono-dominio** comparte
   mucho vocabulario (la empresa, sus cifras y su sostenibilidad se narran con las
   mismas palabras). El contenido de **redes/YouTube** es el más separable.
2. **¿La memoria del OS clusteriza por tema?** El Panel B lo mide sobre los
   vectores **nativos 768-dim** con etiquetas de verdad-de-terreno. El ARI/silhouette
   resultantes son la respuesta empírica — con la salvedad honesta de que el
   tamaño de muestra es pequeño, así que se leen como **indicativos**.
3. **Consecuencia de arquitectura:** como los temas corporativos se solapan en el
   espacio de embeddings, la recuperación puramente semántica es ambigua; por eso
   `manuelita-bot` se apoya además en el **mapa explícito tema→archivo** del
   `system_prompt` y en los **DATOS NÚCLEO**. El t-SNE/UMAP da evidencia visual de
   por qué esa red de seguridad ayuda.

*Reproducible:* re-ejecutar este notebook tras refrescar
`reports/modulo3/_work/openfang.db` con un snapshot de la DB viva.